> Three channel image training

In [ ]:
#| default_exp training_with_config

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import os

In [ ]:
#os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  #

In [ ]:
#| export
from fastcore.all import *
import yaml
from datetime import datetime
from argparse import ArgumentParser, BooleanOptionalAction
from typing import List
from dotenv import load_dotenv

In [ ]:
#| export
from cv_tools.core import *

In [ ]:
#| export
import tensorflow as tf
import tensorflow_addons as tfa 
from tensorflow.keras import backend as K
import mlflow
import mlflow.tensorflow

In [ ]:
tf.config.set_visible_devices([], 'GPU')

In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
#| export
from private_easy_pin_detection.three_channel_model import *
from private_easy_pin_detection.three_channel_dataset import *

In [ ]:
#| export
load_dotenv('/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/.env')

In [ ]:
config_path = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/config_one_c_office.yaml')
#config_path = Path(Path.cwd().parent,'private_easy_pin_detection','config_one_c_office.yaml')
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

In [ ]:
config['dataset']['augmentations']['mixup']

In [ ]:
trn_ds=create_dataset_w_mix(
    config,
    only_mixup=config['dataset']['augmentations']['mixup'],
    only_cutmix=config['dataset']['augmentations']['cut_mix'],
    both=config['dataset']['augmentations']['both'],
    training=True
)


In [ ]:


val_ds=create_dataset_w_mix(
    config,
    training=True,
)

In [ ]:
#trn_dataset = create_dataset_wih(config, training=True)
#val_dataset = create_dataset(config, training=False)

In [ ]:
trn_dataset = create_dataset_w_mix(
    config, training=True,only_mixup=False, only_cutmix=True, both=False)

In [ ]:
image, msk = trn_dataset.take(1).as_numpy_iterator().next()
show_(image[0])
show_(image[1])
show_(image[2])

In [ ]:
img, msk = next(iter(val_dataset))

In [ ]:
model = encoder_decoder(
    input_size=tuple(config['model']['input_shape']),
    n_classes=config['model']['num_classes'],
)

In [ ]:
#| export
def tversky(y_true, y_pred, smooth=1):
    y_true_pos = K.flatten(y_true)
    y_pred_pos = K.flatten(y_pred)
    true_pos = K.sum(y_true_pos * y_pred_pos)
    false_neg = K.sum(y_true_pos * (1-y_pred_pos))
    false_pos = K.sum((1-y_true_pos)*y_pred_pos)
    alpha = 0.8
    return (true_pos + smooth)/(true_pos + alpha*false_neg + (1-alpha)*false_pos + smooth)

def tversky_loss(y_true, y_pred):
    return 1 - tversky(y_true,y_pred)

def focal_tversky_loss_r(y_true,y_pred):
    pt_1 = tversky(y_true, y_pred)
    gamma = 2
    return K.pow((1-pt_1), gamma)

def dice_coefficient(y_true, y_pred):
    smooth = 1e-6  # To avoid division by zero
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)

def focal_loss(y_true, y_pred, gamma=2., alpha=0.25):
    y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon())
    y_true = tf.cast(y_true, tf.float32)
    loss = -y_true * alpha * tf.pow((1 - y_pred), gamma) * tf.math.log(y_pred) - (1 - y_true) * (1 - alpha) * tf.pow(y_pred, gamma) * tf.math.log(1 - y_pred)
    return tf.reduce_mean(loss)

def combined_loss(y_true, y_pred, alpha=0.5, gamma=2.):
    return alpha * focal_loss(y_true, y_pred, gamma) + (1 - alpha) * dice_loss(y_true, y_pred)


class BasnetLoss(tf.keras.losses.Loss):
    """BASNet hybrid loss."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.smooth = 1.0e-9

        # Binary Cross Entropy loss.
        self.cross_entropy_loss = focal_tversky_loss_r
        # Structural Similarity Index value.
        self.ssim_value = tf.image.ssim
        #  Jaccard / IoU loss.
        self.iou_value = self.calculate_iou

    def calculate_iou(
        self,
        y_true,
        y_pred,
    ):
        """Calculate intersection over union (IoU) between images."""
        intersection = K.sum(K.abs(y_true * y_pred), axis=[1, 2, 3])
        union = K.sum(y_true, [1, 2, 3]) + K.sum(y_pred, [1, 2, 3])
        union = union - intersection
        return K.mean(
            (intersection + self.smooth) / (union + self.smooth), axis=0
        )

    def call(self, y_true, y_pred):
        #cross_entropy_loss = self.cross_entropy_loss(y_true, y_pred)
        cross_entropy_loss = focal_tversky_loss_r(y_true, y_pred)

        ssim_value = self.ssim_value(y_true, y_pred, max_val=1)
        ssim_loss = K.mean(1 - ssim_value + self.smooth, axis=0)

        iou_value = self.iou_value(y_true, y_pred)
        iou_loss = 1 - iou_value

        # Add all three losses.
        return cross_entropy_loss + ssim_loss + iou_loss

In [ ]:
#learning_rate = 0.001
weight_decay_rate = 0.0001
initial_learning_rate = 0.002
total_steps = 200  # Assuming one step per epoch

# Define the cosine annealing learning rate schedule
learning_rate_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=initial_learning_rate,
    first_decay_steps=100,  # Decay for the first 100 epochs
    t_mul=2.0,  # Restart period is 2 times the first decay period
    m_mul=0.9,  # Minimum learning rate is 0.9 times the initial learning rate
    alpha=0.1,  # Minimum learning rate value is 0.1 times the initial learning rate
    name=None
)

In [ ]:
optimizer = tfa.optimizers.AdamW(
            learning_rate=1e-3, 
            weight_decay=1e-5, 
            clipnorm=1.0) 

In [ ]:
class BasnetLoss(tf.keras.losses.Loss):
    """BASNet hybrid loss."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.smooth = 1.0e-9

        # Binary Cross Entropy loss.
        self.cross_entropy_loss = focal_tversky_loss_r
        # Structural Similarity Index value.
        self.ssim_value = tf.image.ssim
        #  Jaccard / IoU loss.
        self.iou_value = self.calculate_iou

    def calculate_iou(
        self,
        y_true,
        y_pred,
    ):
        """Calculate intersection over union (IoU) between images."""
        intersection = K.sum(K.abs(y_true * y_pred), axis=[1, 2, 3])
        union = K.sum(y_true, [1, 2, 3]) + K.sum(y_pred, [1, 2, 3])
        union = union - intersection
        return K.mean(
            (intersection + self.smooth) / (union + self.smooth), axis=0
        )

    def call(self, y_true, y_pred):
        #cross_entropy_loss = self.cross_entropy_loss(y_true, y_pred)
        cross_entropy_loss = focal_tversky_loss_r(y_true, y_pred)

        ssim_value = self.ssim_value(y_true, y_pred, max_val=1)
        ssim_loss = K.mean(1 - ssim_value + self.smooth, axis=0)

        iou_value = self.iou_value(y_true, y_pred)
        iou_loss = 1 - iou_value

        # Add all three losses.
        return cross_entropy_loss + ssim_loss + iou_loss

In [ ]:
loss_fn = tf.keras.losses.BinaryFocalCrossentropy(
    from_logits=False,
    gamma=2.0,
    alpha=0.25,
    label_smoothing=0,
    reduction=tf.keras.losses.Reduction.AUTO,
    name='binary_focal_crossentropy'
)

In [ ]:
#| export
def get_data(
    config:dict
    ):

    trn_ds=create_dataset_w_mix(
        config,
        only_mixup=config['dataset']['augmentations']['mixup'],
        only_cutmix=config['dataset']['augmentations']['cut_mix'],
        both=config['dataset']['augmentations']['both'],
        training=True
    )

    val_ds=create_dataset_w_mix(
        config,
        training=False,
        only_mixup=False,
        only_cutmix=False,
        both=False
    )
    return trn_ds, val_ds


In [ ]:
#| export
def get_model(
        config:dict,
        ):
    if not config['model']['pretrained']:
        if config['model']['architecture'] == 'UNet':
            model = encoder_decoder(
                input_size=tuple(config['model']['input_shape']),
                n_classes=config['model']['num_classes'],
            )
        elif config['model']['architecture'] == 'Unet256_Min350Pixel':
            print(f'Patch size is 256')
            model = encoder_decoder_256(
                input_size=tuple(config['model']['input_shape']),
                n_classes=config['model']['num_classes'])
        else:
            model = encode_128_decoder_128(
                input_size=tuple(config['model']['input_shape']),
                n_classes=config['model']['num_classes'],
        )
        
        #print(loaded_model.summary())
       

    else:
        optimizer = tfa.optimizers.AdamW(
                learning_rate=1e-3, 
                weight_decay=1e-5, 
                clipnorm=1.0) 
        model = tf.keras.models.load_model(
            config['model']['pretrained_model_path'],
                        custom_objects={
                'optimizer':optimizer,
                'BasnetLoss':BasnetLoss,
                'focal_tversky_loss_r':focal_tversky_loss_r}
                )
        
        print('pretrained model is used')
        if config['model']['num_trainable_layers'] > 0: #we want some layers only
            for i in model.layers[:config['model']['num_trainable_layers']]:
                i.trainable=False
            
            print(f"number of layers is freezed = {config['model']['num_trainable_layers']}")

    return model

In [ ]:
#| export
def model_training(
    config:dict,
    model:tf.keras.Model,
    trn_ds,
    val_ds,
    loss_fn,
    optimizer,
    verbose:int=1,
    experiment_name:str='easy_pin_detectionV02',
    ):

    MODEL_EXPERIMENT_NAME=experiment_name
    mlflow.set_experiment(MODEL_EXPERIMENT_NAME)

    trn_images = len(Path(config['dataset']['trn_im_path']).ls(file_exts=['.png', '.jpg', '.jpeg', '.tif', '.tiff']))
    val_images = len(Path(config['dataset']['val_im_path']).ls(file_exts=['.png', '.jpg', '.jpeg', '.tif', '.tiff']))

    # Calculate steps per epoch and validation steps
    config['training']['steps_per_epoch'] = trn_images // config['training']['batch_size']
    config['training']['validation_steps'] = val_images // config['training']['batch_size']

    # If there are leftover images not covered by full batches, add an extra step:
    if trn_images % config['training']['batch_size'] > 0:
        config['training']['steps_per_epoch'] += 1

    if val_images % config['training']['batch_size'] > 0:
        config['training']['validation_steps'] += 1

    current_datetime = datetime.now()
    time = current_datetime.strftime("%H_%M_%S")
    Path(config['model']['save_path']).mkdir(parents=True, exist_ok=True)
    model_save_path=config['model']['save_path']

    model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
        monitor='val_foreground',
        filepath=f'{model_save_path}/time_{time}_val_frGrnd'+'{val_foreground:.4f}_epoch_{epoch}.h5', 
        save_best_only=True,
        save_freq='epoch',
        mode='max',  # Assuming you want to maximize the foreground IoU; change to 'min' if you want to minimize something like loss
    )

    

    with mlflow.start_run(): 
        weight_decay_rate = config['training']['weight_decay']
        initial_learning_rate =config['training']['initial_learning_rate']
        total_steps = 200  # Assuming one step per epoch

        # Define the cosine annealing learning rate schedule
        learning_rate_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
            initial_learning_rate=initial_learning_rate,
            first_decay_steps=config['training']['first_decay_steps'], # Decay for the first 100 epochs
            t_mul=config['training']['t_mul'],  # Restart period is 2 times the first decay period
            m_mul=config['training']['m_mul'],  # Minimum learning rate is 0.9 times the initial learning rate
            alpha=config['training']['alpha'],  # Minimum learning rate value is 0.1 times the initial learning rate
            name=None
        )
        model.compile(
                    loss=loss_fn,
                    optimizer=optimizer,
                    metrics =[
                        tf.keras.metrics.BinaryIoU(
                            name='foreground', 
                            target_class_ids=[1], 
                            threshold=0.5)]
                            )
        callbacks = [model_checkpoint]
        history = model.fit(
            trn_ds,
            validation_data=val_ds,
            steps_per_epoch=config['training']['steps_per_epoch'],
            validation_steps=config['training']['validation_steps'],
            epochs=config['training']['epochs'],
            callbacks=callbacks
        )
        for epoch in range(config['training']['epochs']):
            mlflow.log_metric('train_loss', history.history['loss'][epoch], step=epoch)
            mlflow.log_metric('val_loss', history.history['val_loss'][epoch], step=epoch)
            mlflow.log_metric('foreground', history.history['foreground'][epoch], step=epoch)
            mlflow.log_metric('val_foreground', history.history['val_foreground'][epoch], step=epoch)


In [ ]:
#| export
def parse_args():
    parser = ArgumentParser()
    parser.add_argument(
        '--config_path', type=str, required=True, help='Path to the config file'
    )
    parser.add_argument(
        '--experiment_name', type=str, required=True, help='Name of the experiment'
    )
    return parser.parse_args()

In [ ]:
#| export
def full_pipeline(config_path:str,experiment_name:str):
    with open(config_path, 'r') as file:
        config = yaml.safe_load(file)

    trn_ds, val_ds = get_data(config)

    # LossFunction
    if config['training']['loss'] == 'BasNetLoss':
        loss_fn = BasnetLoss()
    else:
        loss_fn = tf.keras.losses.BinaryFocalCrossentropy(
            from_logits=False,
            gamma=2.0,
            alpha=0.25,
            label_smoothing=0,
            reduction=tf.keras.losses.Reduction.AUTO,
            name='binary_focal_crossentropy'
        )

    # Optimizer
    optimizer = tfa.optimizers.AdamW(
                learning_rate=1e-3, 
                weight_decay=1e-5, 
                clipnorm=1.0) 

    # Model
    model = get_model(config)
    #  Training
    model_training(
        config, 
        model, 
        trn_ds, 
        val_ds, 
        loss_fn, 
        optimizer, 
        experiment_name='easy_pin_detectionV02')

In [ ]:
#| export
def main_():
    args = parse_args()
    full_pipeline(args.config_path, args.experiment_name)

In [ ]:
#| hide
model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=[
        tf.keras.metrics.BinaryIoU(
            name='foreground', 
            target_class_ids=[1], 
            threshold=0.5),
        dice_coefficient]
)

current_datetime = datetime.now()
time = current_datetime.strftime("%H_%M_%S")
Path(config['model']['save_path']).mkdir(parents=True, exist_ok=True)

model_save_path = config['model']['save_path'] 
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    monitor='val_foreground',
    filepath=f'{model_save_path}/time_{time}_val_frGrnd'+'{val_foreground:.4f}_epoch_{epoch}.h5', 
    save_best_only=True,
    save_freq='epoch',
    mode='max',  # Assuming you want to maximize the foreground IoU; change to 'min' if you want to minimize something like loss
)

trn_images = len(Path(config['dataset']['trn_im_path']).ls(file_exts=['.png', '.jpg', '.jpeg', '.tif', '.tiff']))
val_images = len(Path(config['dataset']['val_im_path']).ls(file_exts=['.png', '.jpg', '.jpeg', '.tif', '.tiff']))
# Calculate steps per epoch and validation steps
config['training']['steps_per_epoch'] = trn_images // config['training']['batch_size']
config['training']['validation_steps'] = val_images // config['training']['batch_size']

# If there are leftover images not covered by full batches, add an extra step:
if trn_images % config['training']['batch_size'] > 0:
    config['training']['steps_per_epoch'] += 1

if val_images % config['training']['batch_size'] > 0:
    config['training']['validation_steps'] += 1


callbacks = [model_checkpoint]
his = model.fit(
    trn_dataset,
    validation_data=val_dataset,
    steps_per_epoch=config['training']['steps_per_epoch'],
    validation_steps=config['training']['validation_steps'],
    epochs=config['training']['epochs'],
    callbacks=callbacks
)

In [ ]:
#| export
if __name__ == '__main__':
    main_()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('14_three_channel_training.ipynb')